# PROF-GRPO Training Notebook (Using Official Algorithm)

This notebook implements PROF-GRPO using the **official algorithm** from the Amazon PROF-GRPO repository.

**Paper**: "PROF: Process-Consistency Filter for GRPO" (Ye et al.)

## Offline Usage Instructions (Kaggle)

To run this notebook without internet access on Kaggle, you need to:

### Pre-download Models (run locally first):
```python
from huggingface_hub import snapshot_download

# Policy Model
snapshot_download("Qwen/Qwen2.5-Math-1.5B-Instruct", local_dir="./models/qwen-math-1.5b")

# Process Reward Model
snapshot_download("Qwen/Qwen2.5-Math-PRM-7B", local_dir="./models/qwen-prm-7b")
```

### Pre-download Dataset:
```python
from datasets import load_dataset
ds = load_dataset("AI-MO/NuminaMath-CoT", split="train[:5000]")
ds.save_to_disk("./datasets/numina-math-cot-5k")
```

### Upload to Kaggle:
1. Zip the `models/` and `datasets/` folders
2. Upload as Kaggle datasets
3. Add them as input to your notebook
4. Set `OFFLINE_MODE = True` below and adjust paths

---

In [ ]:
# ============================================================
# CELL 0: Configuration
# ============================================================

# Set to True when running without internet on Kaggle
OFFLINE_MODE = False

# Paths for offline mode (matches download_for_offline.py output)
OFFLINE_ASSETS = "/kaggle/input/prof-grpo-offline-assets"
OFFLINE_POLICY_MODEL_PATH = f"{OFFLINE_ASSETS}/models/qwen-math-1.5b"
OFFLINE_PRM_MODEL_PATH = f"{OFFLINE_ASSETS}/models/qwen-prm-7b"
OFFLINE_DATASET_PATH = f"{OFFLINE_ASSETS}/datasets/numina-math-cot-5k"

# Online paths (HuggingFace)
ONLINE_POLICY_MODEL = "Qwen/Qwen2.5-Math-1.5B-Instruct"
ONLINE_PRM_MODEL = "Qwen/Qwen2.5-Math-PRM-7B"
ONLINE_DATASET = "AI-MO/NuminaMath-CoT"

# PROF hyperparameters (from official repo)
PROF_CONFIG = {
    "n_rollouts": 8,           # Number of rollouts per prompt
    "prof_filter": 4,          # Number of trajectories to REMOVE (keeps n - prof_filter)
    "len_step_reward_coef": 10.0,  # Penalty for steps < 2 or > 30
    "gamma": 0.99,             # PRM discount factor
}

# Training config
# FOR TESTING: use small values (500 samples, 100 steps)
# FOR REAL TRAINING: use larger values (5000 samples, 500 steps)
TRAIN_CONFIG = {
    "num_train_samples": 5000,  # Number of training samples from dataset
    "max_steps": 500,           # Training steps
    "batch_size": 2,            # Prompts per batch (use 4 for L4 GPU)
    "learning_rate": 1e-6,
    "lora_r": 16,
    "lora_alpha": 32,
}

print("Configuration set.")
print(f"  OFFLINE_MODE: {OFFLINE_MODE}")
print(f"  PROF n_rollouts: {PROF_CONFIG['n_rollouts']}")
print(f"  PROF filter removes: {PROF_CONFIG['prof_filter']} (keeps {PROF_CONFIG['n_rollouts'] - PROF_CONFIG['prof_filter']})")
print(f"  Training samples: {TRAIN_CONFIG['num_train_samples']}")
print(f"  Max steps: {TRAIN_CONFIG['max_steps']}")
print(f"  Batch size: {TRAIN_CONFIG['batch_size']}")

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
import sys
print(f"Python: {sys.version}")

# Check if running in offline mode (OFFLINE_MODE must be defined in Cell 0)
try:
    if OFFLINE_MODE:
        print("OFFLINE MODE: Installing from local wheels...")
        !pip install /kaggle/input/prof-grpo-offline-assets/wheels/*.whl --no-index --find-links /kaggle/input/prof-grpo-offline-assets/wheels/ -q
        print("Offline installation complete!")
    else:
        raise NameError  # Fall through to online install
except NameError:
    # Online installation
    print("\n=== Installing core packages ===")
    !pip install -U datasets scipy transformers accelerate peft bitsandbytes -q

    print("\n=== Installing TRL ===")
    !pip install "trl<0.9.0" -q

    print("\n=== Installing Unsloth ===")
    !pip install unsloth -q

print("\n" + "="*60)
print("DEPENDENCIES INSTALLED")
print("="*60)

# Verify installations
import torch
import transformers
import unsloth
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset, Dataset
import numpy as np
import re
from collections import defaultdict

print(f"\nVersions:")
print(f"  PyTorch: {torch.__version__}")
print(f"  Transformers: {transformers.__version__}")
print(f"  Unsloth: {unsloth.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# ============================================================
# CELL 2: Official PROF Algorithm Functions
# ============================================================
# These are extracted directly from the official Amazon PROF-GRPO repository:
# https://github.com/amazon-science/PROF-GRPO
# File: prof_grpo/prof_grpo_both/prof_ray_trainer.py

import torch
import numpy as np
from collections import defaultdict


def step_level_segment(response, sep_token_ids, pad_token_id):
    """
    Segment the response into step-level chunks using \n\n as the delimiter.
    Adjust segments to align with log_probs, which predict the next token.
    
    From official PROF-GRPO repo: prof_ray_trainer.py:1271-1317
    """
    sep_len = len(sep_token_ids)
    sep_token_tensor = torch.tensor(sep_token_ids, device=response.device)

    L = response.shape[0]
    non_pad_indices = (response != pad_token_id).nonzero(as_tuple=True)[0]
    if len(non_pad_indices) == 0:
        return []

    last_non_pad = non_pad_indices[-1].item()

    segments = []
    last_seg_begin_idx = 0

    j = 0
    while j <= last_non_pad - sep_len + 1:
        if torch.equal(response[j : j + sep_len], sep_token_tensor):
            start_idx = last_seg_begin_idx
            end_idx = j - 1  # align with log_probs
            if start_idx <= end_idx:
                segments.append([start_idx, end_idx])
            last_seg_begin_idx = j + sep_len
            j += sep_len
        else:
            j += 1

    # add final segment
    if last_seg_begin_idx <= last_non_pad:
        segments.append([last_seg_begin_idx, last_non_pad])

    # Post-process: merge segments with length <= 4 into the previous segment
    if len(segments) <= 1:
        return segments
    merged_segments = [segments[0]]
    for i in range(1, len(segments)):
        seg_start, seg_end = segments[i]
        seg_len = seg_end - seg_start + 1
        if seg_len <= 4 and len(merged_segments) > 0:
            # Merge this segment into the previous one
            merged_segments[-1][1] = seg_end
        else:
            merged_segments.append([seg_start, seg_end])
    return merged_segments


def compute_verify_step_level_score(scores: torch.Tensor, final_rewards: torch.Tensor, gamma: float):
    """
    Computes a step-level consistency score.
    
    From official PROF-GRPO repo: prof_ray_trainer.py:1377-1433
    
    Args:
        scores (torch.Tensor): 2D tensor of PRM scores, where -99.0 indicates padding.
        final_rewards (torch.Tensor): 1D tensor of final rewards (0 or 1).
        gamma (float): Discount factor (not used in this calculation).

    Returns:
        np.ndarray: 1D array of computed step-level scores (scores_ic).
        np.ndarray: 1D array of lengths of valid steps for each sample.
    """
    # Create a boolean mask to identify valid scores (not padding)
    valid_mask = (scores != -99.0)
    
    # Calculate the number of valid steps for each sample in the batch
    len_segs = valid_mask.sum(dim=1)
    
    # Calculate the single mean score across all valid steps in the batch
    score_uns_mean = scores[valid_mask].mean()
    
    # Calculate "score_processed": subtract mean and multiply by 2
    score_processed = (scores - score_uns_mean) * 2
    
    # Create scaling factor: +1 for final_reward=1, -1 for final_reward=0
    final_rewards_scaled = (2 * final_rewards - 1).unsqueeze(1)
    
    # Scale the processed scores
    scaled_values = score_processed * final_rewards_scaled
    
    # Zero out padded values
    scaled_values_masked = scaled_values * valid_mask
    
    # Calculate mean of scaled values for each sample
    sum_of_scaled_values = torch.sum(scaled_values_masked, dim=1)
    safe_len_segs = torch.clamp(len_segs, min=1)
    scores_ic = sum_of_scaled_values / safe_len_segs
    
    # Handle edge cases for all-padding samples
    scores_ic[len_segs == 0] = 0.0

    return scores_ic.cpu().numpy(), len_segs.cpu().numpy()


def filter_trajectories_official(
    uids: list,
    scores_ic: np.ndarray,
    len_segs: np.ndarray,
    outcome_rewards: list,
    n_remove_per_group: int = 4,
    len_step_reward_coef: float = 10.0
) -> list:
    """
    Official PROF filtering algorithm.
    
    From official PROF-GRPO repo: prof_ray_trainer.py:1437-1518
    
    Args:
        uids: List of unique IDs for each prompt (groups responses by prompt)
        scores_ic: Consistency scores from compute_verify_step_level_score
        len_segs: Number of steps per response
        outcome_rewards: Binary rewards (0 or 1) for each response
        n_remove_per_group: Number to remove from each group (prof_filter param)
        len_step_reward_coef: Penalty coefficient for step length violations
        
    Returns:
        List of indices to keep
    """
    prompt_uid2idx_score = defaultdict(list)
    
    for idx, (uid, prm_val, len_seg, gt) in enumerate(zip(
        uids, scores_ic, len_segs, outcome_rewards
    )):
        # Apply step length penalty
        prm_critic = prm_val - len_step_reward_coef * ((len_seg < 2) or (len_seg > 30))
        # Store: (original index, critic score, ground truth reward)
        prompt_uid2idx_score[uid].append((idx, prm_critic, gt))

    kept_traj_idxs = []
    
    for uid, idx_score_list in prompt_uid2idx_score.items():
        n_total = len(idx_score_list)
        
        if n_total == 0:
            continue

        n_remove = min(n_remove_per_group, n_total)

        if n_remove <= 0:
            kept_traj_idxs.extend([item[0] for item in idx_score_list])
            continue

        # Separate into positive and negative samples
        pos_samples = [item for item in idx_score_list if item[2] == 1]
        neg_samples = [item for item in idx_score_list if item[2] == 0]

        delta = len(pos_samples) - len(neg_samples)
        n_pos_to_remove = min(n_remove, round((delta + n_remove) / 2))

        kept_pos_samples = list(pos_samples)
        kept_neg_samples = list(neg_samples)

        if n_pos_to_remove > 0:
            # Sort positive by prm_critic (ascending) to find lowest to remove
            pos_samples.sort(key=lambda x: x[1])
            
            num_removable_pos = min(n_pos_to_remove, len(pos_samples))
            pos_to_remove_indices = {item[0] for item in pos_samples[:num_removable_pos]}
            kept_pos_samples = [item for item in pos_samples if item[0] not in pos_to_remove_indices]

            # Remove remaining from negative samples
            n_neg_to_remove = n_remove - num_removable_pos
            if n_neg_to_remove > 0 and len(neg_samples) > 0:
                num_removable_neg = min(n_neg_to_remove, len(neg_samples))
                neg_samples.sort(key=lambda x: x[1])
                neg_to_remove_indices = {item[0] for item in neg_samples[:num_removable_neg]}
                kept_neg_samples = [item for item in neg_samples if item[0] not in neg_to_remove_indices]
        else:
            # Remove from negative samples only
            if n_remove > 0 and len(neg_samples) > 0:
                num_removable_neg = min(n_remove, len(neg_samples))
                neg_samples.sort(key=lambda x: x[1])
                neg_to_remove_indices = {item[0] for item in neg_samples[:num_removable_neg]}
                kept_neg_samples = [item for item in neg_samples if item[0] not in neg_to_remove_indices]

        kept_this_group = kept_pos_samples + kept_neg_samples
        kept_traj_idxs.extend([item[0] for item in kept_this_group])
        
    return kept_traj_idxs


print("Official PROF algorithm functions loaded:")
print("  - step_level_segment()")
print("  - compute_verify_step_level_score()")
print("  - filter_trajectories_official()")

In [ ]:
# ============================================================
# CELL 3: Load Policy Model
# ============================================================
from unsloth import FastLanguageModel
import torch

# Select paths based on mode
if OFFLINE_MODE:
    policy_model_path = OFFLINE_POLICY_MODEL_PATH
    prm_model_path = OFFLINE_PRM_MODEL_PATH
else:
    policy_model_path = ONLINE_POLICY_MODEL
    prm_model_path = ONLINE_PRM_MODEL

print(f"Loading policy model from: {policy_model_path}")
print(f"PRM model path: {prm_model_path}")

# Load policy model with Unsloth
# NOTE: load_in_4bit=False to avoid dtype mismatch errors during backward pass
policy_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=policy_model_path,
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=False,  # IMPORTANT: Must be False to avoid Half/Float mismatch
)

# Add LoRA adapters
policy_model = FastLanguageModel.get_peft_model(
    policy_model,
    r=TRAIN_CONFIG["lora_r"],
    lora_alpha=TRAIN_CONFIG["lora_alpha"],
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing=True,  # Use True, not "unsloth"
    random_state=42,
)

print(f"\nPolicy model loaded with LoRA (r={TRAIN_CONFIG['lora_r']})")
print(f"  Trainable params: {sum(p.numel() for p in policy_model.parameters() if p.requires_grad):,}")
print(f"  VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# ============================================================
# CELL 4: Load PRM (Process Reward Model)
# ============================================================
# Official loading code from Qwen2.5-Math-PRM-7B model card
# https://huggingface.co/Qwen/Qwen2.5-Math-PRM-7B
#
# NOTE: There's a known bug where Qwen2RMConfig is missing pad_token_id.
# We patch it before loading.

from transformers import AutoModel, AutoTokenizer as HFTokenizer, AutoConfig
import torch

print(f"Loading PRM from: {prm_model_path}")
print("(This requires transformers >= 4.40.0)")

# Load tokenizer first
prm_tokenizer = HFTokenizer.from_pretrained(
    prm_model_path, 
    trust_remote_code=True
)

# Load config and patch missing pad_token_id
prm_config = AutoConfig.from_pretrained(prm_model_path, trust_remote_code=True)
if not hasattr(prm_config, 'pad_token_id') or prm_config.pad_token_id is None:
    prm_config.pad_token_id = prm_tokenizer.pad_token_id or 0
    print(f"  Patched missing pad_token_id: {prm_config.pad_token_id}")

# Load model with patched config
prm_model = AutoModel.from_pretrained(
    prm_model_path,
    config=prm_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
).eval()

# Get step separator token ID
STEP_TAG = "<extra_0>"
STEP_TAG_ID = prm_tokenizer.encode(STEP_TAG)[0]

print(f"\nPRM loaded successfully!")
print(f"  Model type: {type(prm_model).__name__}")
print(f"  STEP_TAG_ID (<extra_0>): {STEP_TAG_ID}")

In [ ]:
# ============================================================
# CELL 5: PRM Scoring Functions
# ============================================================
# Official scoring code from Qwen2.5-Math-PRM-7B model card

import torch
import torch.nn.functional as F

def make_step_rewards(logits: torch.Tensor, token_masks: torch.Tensor) -> list:
    """
    Official step reward extraction from Qwen2.5-Math-PRM-7B model card.
    
    Args:
        logits: Model output logits
        token_masks: Boolean mask where True indicates <extra_0> positions
        
    Returns:
        List of lists containing step-level reward probabilities
    """
    probabilities = F.softmax(logits, dim=-1)
    probabilities = probabilities * token_masks.unsqueeze(-1)
    
    all_scores_res = []
    for i in range(probabilities.size(0)):
        sample = probabilities[i]
        # Get non-zero probabilities (at step marker positions)
        positive_probs = sample[sample != 0].view(-1, 2)[:, 1]
        non_zero_elements_list = positive_probs.cpu().tolist()
        all_scores_res.append(non_zero_elements_list if non_zero_elements_list else [0.5])
    return all_scores_res


def compute_prm_scores_batch(prompts: list, completions: list, batch_size: int = 2) -> tuple:
    """
    Compute PRM scores for a batch of responses.
    
    Returns:
        scores: 2D tensor of step scores (padded with -99.0)
        len_segs: 1D array of number of steps per response
    """
    all_step_rewards = []
    
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i+batch_size]
        batch_completions = completions[i:i+batch_size]
        
        # Format using chat template with <extra_0> step markers
        formatted_texts = []
        for p, c in zip(batch_prompts, batch_completions):
            # Split completion into steps and join with <extra_0>
            steps = c.split("\n\n")
            marked_completion = "<extra_0>".join(steps) + "<extra_0>"
            
            # Create message format
            messages = [
                {"role": "system", "content": "Please reason step by step, and put your final answer within \\boxed{}."},
                {"role": "user", "content": p},
                {"role": "assistant", "content": marked_completion},
            ]
            
            # Apply chat template
            try:
                text = prm_tokenizer.apply_chat_template(
                    messages, 
                    tokenize=False, 
                    add_generation_prompt=False
                )
            except:
                # Fallback if chat template fails
                text = f"{p}\n{marked_completion}"
            
            formatted_texts.append(text)
        
        # Tokenize
        inputs = prm_tokenizer(
            formatted_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(prm_model.device)
        
        try:
            with torch.no_grad():
                outputs = prm_model(input_ids=inputs.input_ids)
                
                # Create token mask for <extra_0> positions
                token_masks = (inputs.input_ids == STEP_TAG_ID).to(prm_model.device)
                
                # Extract step rewards using official function
                step_rewards = make_step_rewards(outputs[0], token_masks)
                all_step_rewards.extend(step_rewards)
                
        except Exception as e:
            print(f"  PRM scoring error: {e}")
            # Fallback: return default scores
            for _ in batch_prompts:
                all_step_rewards.append([0.5])
        
        # Memory cleanup
        del inputs
        torch.cuda.empty_cache()
    
    # Pad to create 2D tensor
    max_steps = max(len(r) for r in all_step_rewards) if all_step_rewards else 1
    padded_scores = []
    len_segs = []
    
    for rewards in all_step_rewards:
        len_segs.append(len(rewards))
        padded = rewards + [-99.0] * (max_steps - len(rewards))
        padded_scores.append(padded)
    
    scores_tensor = torch.tensor(padded_scores, dtype=torch.float32)
    len_segs_array = np.array(len_segs)
    
    return scores_tensor, len_segs_array


print("PRM scoring functions loaded (official Qwen implementation):")
print("  - make_step_rewards()")
print("  - compute_prm_scores_batch()")

In [ ]:
# ============================================================
# CELL 6: Outcome Reward Function (ORM)
# ============================================================
# Using the OFFICIAL math scoring code from PROF-GRPO repo:
# verl/utils/reward_score/math.py
# Adapted from EleutherAI lm-evaluation-harness

import re

def last_boxed_only_string(string):
    """Extract the last \\boxed{...} from a string, handling nested braces."""
    idx = string.rfind("\\boxed")
    if "\\boxed " in string:
        return "\\boxed " + string.split("\\boxed ")[-1].split("$")[0]
    if idx < 0:
        idx = string.rfind("\\fbox")
        if idx < 0:
            return None

    i = idx
    right_brace_idx = None
    num_left_braces_open = 0
    while i < len(string):
        if string[i] == "{":
            num_left_braces_open += 1
        if string[i] == "}":
            num_left_braces_open -= 1
            if num_left_braces_open == 0:
                right_brace_idx = i
                break
        i += 1

    if right_brace_idx is None:
        return None
    return string[idx : right_brace_idx + 1]


def remove_boxed(s):
    """Remove \\boxed{} wrapper and return the contents."""
    if s is None:
        return None
    if "\\boxed " in s:
        left = "\\boxed "
        assert s[: len(left)] == left
        return s[len(left):]

    left = "\\boxed{"
    if s[:len(left)] != left:
        return s
    if s[-1] != "}":
        return s
    return s[len(left):-1]


def fix_fracs(string):
    substrs = string.split("\\frac")
    new_str = substrs[0]
    if len(substrs) > 1:
        substrs = substrs[1:]
        for substr in substrs:
            new_str += "\\frac"
            if len(substr) == 0:
                continue
            if substr[0] == "{":
                new_str += substr
            else:
                try:
                    assert len(substr) >= 2
                except:
                    return string
                a = substr[0]
                b = substr[1]
                if b != "{":
                    if len(substr) > 2:
                        post_substr = substr[2:]
                        new_str += "{" + a + "}{" + b + "}" + post_substr
                    else:
                        new_str += "{" + a + "}{" + b + "}"
                else:
                    if len(substr) > 2:
                        post_substr = substr[2:]
                        new_str += "{" + a + "}" + b + post_substr
                    else:
                        new_str += "{" + a + "}" + b
    return new_str


def fix_sqrt(string):
    if "\\sqrt" not in string:
        return string
    splits = string.split("\\sqrt")
    new_string = splits[0]
    for split in splits[1:]:
        if len(split) == 0:
            new_string += "\\sqrt"
            continue
        if split[0] != "{":
            a = split[0]
            new_substr = "\\sqrt{" + a + "}" + split[1:]
        else:
            new_substr = "\\sqrt" + split
        new_string += new_substr
    return new_string


def fix_a_slash_b(string):
    if len(string.split("/")) != 2:
        return string
    a = string.split("/")[0]
    b = string.split("/")[1]
    try:
        a = int(a)
        b = int(b)
        assert string == "{}/{}".format(a, b)
        new_string = "\\frac{" + str(a) + "}{" + str(b) + "}"
        return new_string
    except:
        return string


def remove_right_units(string):
    if "\\text{ " in string:
        splits = string.split("\\text{ ")
        return splits[0]
    return string


def strip_string(string):
    """Normalize a LaTeX math string for comparison."""
    string = string.replace("\n", "")
    string = string.replace("\\!", "")
    string = string.replace("\\\\", "\\")
    string = string.replace("tfrac", "frac")
    string = string.replace("dfrac", "frac")
    string = string.replace("\\left", "")
    string = string.replace("\\right", "")
    string = string.replace("^{\\circ}", "")
    string = string.replace("^\\circ", "")
    string = string.replace("\\$", "")
    string = remove_right_units(string)
    string = string.replace("\\%", "")
    string = string.replace("\%", "")
    string = string.replace(" .", " 0.")
    string = string.replace("{.", "{0.")
    if len(string) == 0:
        return string
    if string[0] == ".":
        string = "0" + string
    if len(string.split("=")) == 2 and len(string.split("=")[0]) <= 2:
        string = string.split("=")[1]
    string = fix_sqrt(string)
    string = string.replace(" ", "")
    string = fix_fracs(string)
    if string == "0.5":
        string = "\\frac{1}{2}"
    string = fix_a_slash_b(string)
    return string


def is_equiv(str1, str2):
    """Check if two math strings are equivalent after normalization."""
    if str1 is None and str2 is None:
        return True
    if str1 is None or str2 is None:
        return False
    try:
        ss1 = strip_string(str1)
        ss2 = strip_string(str2)
        return ss1 == ss2
    except Exception:
        return str1 == str2


def compute_score(solution_str, ground_truth) -> float:
    """
    Official PROF-GRPO reward scoring function.
    Returns 1.0 if correct, 0.0 if incorrect.
    """
    try:
        # Extract answer from model's response
        string_in_last_boxed = last_boxed_only_string(solution_str)
        if string_in_last_boxed is not None:
            answer = remove_boxed(string_in_last_boxed)
            
            # Also extract answer from ground truth if it's in boxed format
            gt_boxed = last_boxed_only_string(ground_truth)
            if gt_boxed is not None:
                ground_truth = remove_boxed(gt_boxed)
            
            if is_equiv(answer, ground_truth):
                return 1.0
    except Exception as e:
        pass
    return 0.0


# Alias for backward compatibility
def compute_outcome_reward(completion: str, ground_truth: str) -> float:
    return compute_score(completion, ground_truth)


print("Official math scoring functions loaded:")
print("  - last_boxed_only_string() - handles nested braces")
print("  - strip_string() - normalizes LaTeX")
print("  - is_equiv() - compares normalized answers")
print("  - compute_score() - main reward function")

# Test
test_cases = [
    ("The answer is \\boxed{42}", "42", 1.0),
    ("So we get \\boxed{\\frac{1}{2}}", "0.5", 1.0),
    ("Therefore \\boxed{\\frac{13}{9}}", "\\boxed{\\frac{13}{9}}", 1.0),
    ("\\boxed{x = 5}", "5", 1.0),
    ("No boxed answer here", "42", 0.0),
]

print("\nTest cases:")
for completion, gt, expected in test_cases:
    result = compute_score(completion, gt)
    status = "✓" if result == expected else "✗"
    print(f"  {status} compute_score('{completion[:30]}...', '{gt}') = {result}")

In [ ]:
# ============================================================
# CELL 7: PROF Reward Wrapper (Combines PRM + ORM + Filtering)
# ============================================================
import uuid

class PROFRewardSystem:
    """
    Complete PROF reward system that:
    1. Computes outcome rewards (ORM)
    2. Computes PRM step scores
    3. Computes consistency scores
    4. Applies PROF filtering
    5. Returns filtered rewards
    """
    
    def __init__(
        self,
        n_rollouts: int = 8,
        prof_filter: int = 4,
        len_step_reward_coef: float = 10.0,
        gamma: float = 0.99
    ):
        self.n_rollouts = n_rollouts
        self.prof_filter = prof_filter
        self.len_step_reward_coef = len_step_reward_coef
        self.gamma = gamma
        
        # Cache for ground truth answers (set before training)
        self.ground_truths = {}
        
        # Track current batch info
        self.current_batch_id = None
        self.batch_scores = {}
        
    def set_ground_truths(self, prompts: list, answers: list):
        """Cache ground truth answers for reward computation."""
        for p, a in zip(prompts, answers):
            self.ground_truths[p] = a
            
    def compute_rewards(
        self,
        prompts: list,
        completions: list,
        use_filtering: bool = True
    ) -> list:
        """
        Compute PROF rewards with optional filtering.
        
        Args:
            prompts: List of prompt strings
            completions: List of completion strings
            use_filtering: Whether to apply PROF filtering
            
        Returns:
            List of reward floats (filtered samples get 0.0)
        """
        n_samples = len(prompts)
        
        # Step 1: Compute outcome rewards
        outcome_rewards = []
        for p, c in zip(prompts, completions):
            gt = self.ground_truths.get(p, "")
            reward = compute_outcome_reward(c, gt)
            outcome_rewards.append(reward)
        
        if not use_filtering or n_samples < self.n_rollouts:
            # Without filtering, just return outcome rewards
            return [float(r) for r in outcome_rewards]
        
        # Step 2: Compute PRM scores
        scores_tensor, len_segs = compute_prm_scores_batch(prompts, completions)
        
        # Step 3: Compute consistency scores
        final_rewards_tensor = torch.tensor(outcome_rewards, dtype=torch.float32)
        scores_ic, _ = compute_verify_step_level_score(
            scores_tensor, final_rewards_tensor, self.gamma
        )
        
        # Step 4: Group by prompt (create UIDs for grouping)
        # Assume prompts are repeated n_rollouts times each
        n_prompts = n_samples // self.n_rollouts
        uids = []
        for i in range(n_prompts):
            uid = str(uuid.uuid4())
            uids.extend([uid] * self.n_rollouts)
        
        # Handle leftover samples (shouldn't happen with proper batching)
        while len(uids) < n_samples:
            uids.append(str(uuid.uuid4()))
        
        # Step 5: Apply PROF filtering
        kept_indices = filter_trajectories_official(
            uids=uids,
            scores_ic=scores_ic,
            len_segs=len_segs,
            outcome_rewards=outcome_rewards,
            n_remove_per_group=self.prof_filter,
            len_step_reward_coef=self.len_step_reward_coef
        )
        
        # Step 6: Create filtered reward list
        kept_set = set(kept_indices)
        filtered_rewards = []
        for i in range(n_samples):
            if i in kept_set:
                filtered_rewards.append(float(outcome_rewards[i]))
            else:
                # Filtered out samples get 0 reward (neutral)
                filtered_rewards.append(0.0)
        
        # Log filtering stats
        n_kept = len(kept_indices)
        print(f"  PROF filtering: {n_samples} -> {n_kept} samples ({n_samples - n_kept} filtered)")
        
        return filtered_rewards


# Create global PROF system instance
prof_system = PROFRewardSystem(
    n_rollouts=PROF_CONFIG["n_rollouts"],
    prof_filter=PROF_CONFIG["prof_filter"],
    len_step_reward_coef=PROF_CONFIG["len_step_reward_coef"],
    gamma=PROF_CONFIG["gamma"]
)

print("PROF Reward System initialized:")
print(f"  n_rollouts: {prof_system.n_rollouts}")
print(f"  prof_filter: {prof_system.prof_filter} (removes this many per group)")
print(f"  len_step_reward_coef: {prof_system.len_step_reward_coef}")

In [ ]:
# ============================================================
# CELL 8: TRL-Compatible Reward Function
# ============================================================

# Create a simple function wrapper for TRL's GRPOTrainer
def prof_reward_fn(prompts: list, completions: list, **kwargs) -> list:
    """
    TRL-compatible reward function using PROF algorithm.
    """
    return prof_system.compute_rewards(
        prompts=prompts,
        completions=[c[0]["content"] if isinstance(c, list) else c for c in completions],
        use_filtering=True
    )

# Also create a baseline reward function (no filtering) for comparison
def baseline_reward_fn(prompts: list, completions: list, **kwargs) -> list:
    """
    Baseline reward function (no PROF filtering).
    """
    return prof_system.compute_rewards(
        prompts=prompts,
        completions=[c[0]["content"] if isinstance(c, list) else c for c in completions],
        use_filtering=False
    )

print("Reward functions created:")
print("  - prof_reward_fn (with PROF filtering)")
print("  - baseline_reward_fn (no filtering)")

In [ ]:
# ============================================================
# CELL 9: Load Dataset
# ============================================================
from datasets import load_dataset, Dataset, load_from_disk

if OFFLINE_MODE:
    print(f"Loading dataset from disk: {OFFLINE_DATASET_PATH}")
    raw_dataset = load_from_disk(OFFLINE_DATASET_PATH)
else:
    print(f"Loading dataset from HuggingFace: {ONLINE_DATASET}")
    raw_dataset = load_dataset(ONLINE_DATASET, split=f"train[:{TRAIN_CONFIG['num_train_samples']}]")

print(f"Dataset size: {len(raw_dataset)}")
print(f"Columns: {raw_dataset.column_names}")

# Format dataset for GRPO
def format_for_grpo(examples):
    formatted = []
    for i in range(len(examples["problem"])):
        prompt = f"Solve this math problem step by step. Put your final answer in \\boxed{{}}:\n\n{examples['problem'][i]}"
        formatted.append({
            "prompt": prompt,
            "ground_truth": examples["solution"][i] if "solution" in examples else ""
        })
    return {"prompt": [f["prompt"] for f in formatted], 
            "ground_truth": [f["ground_truth"] for f in formatted]}

dataset = raw_dataset.map(format_for_grpo, batched=True, remove_columns=raw_dataset.column_names)

# Cache ground truths for reward computation
for item in dataset:
    prof_system.ground_truths[item["prompt"]] = item["ground_truth"]

print(f"\nFormatted dataset:")
print(f"  Size: {len(dataset)}")
print(f"  Columns: {dataset.column_names}")
print(f"  Ground truths cached: {len(prof_system.ground_truths)}")

In [ ]:
import torch

# Check what's actually available
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

grpo_config = GRPOConfig(
    output_dir="./prof-grpo-output",
    
    # Training parameters
    num_train_epochs=1,
    max_steps=TRAIN_CONFIG["max_steps"],
    per_device_train_batch_size=TRAIN_CONFIG["batch_size"],
    learning_rate=TRAIN_CONFIG["learning_rate"],
    
    # Generation parameters
    max_completion_length=512,
    num_generations=PROF_CONFIG["n_rollouts"],
    
    # GRPO specific
    temperature=0.7,
    
    # Logging
    logging_steps=10,
    save_steps=50,
    
    # Memory optimization
    gradient_accumulation_steps=4,
    fp16=not torch.cuda.is_bf16_supported(),   # fallback to fp16 if no bf16
    bf16=torch.cuda.is_bf16_supported(),        # only enable if supported
    
    report_to="none" if OFFLINE_MODE else "wandb",
)

print("GRPO Configuration:")
print(f"  max_steps: {grpo_config.max_steps}")
print(f"  batch_size: {grpo_config.per_device_train_batch_size}")
print(f"  learning_rate: {grpo_config.learning_rate}")

In [ ]:
# ============================================================
# CELL 11: Create Trainer
# ============================================================

trainer = GRPOTrainer(
    model=policy_model,
    args=grpo_config,
    train_dataset=dataset,
    tokenizer=tokenizer,
    reward_funcs=prof_reward_fn,  # Use PROF reward function
)

print("PROF-GRPO Trainer created.")
print("\nReady to train! Run the next cell to start training.")

In [ ]:
# ============================================================
# CELL 12: Train!
# ============================================================
import time

print("="*60)
print("STARTING PROF-GRPO TRAINING")
print("="*60)
print(f"\nConfiguration:")
print(f"  - Policy: {policy_model_path}")
print(f"  - PRM: {prm_model_path}")
print(f"  - Dataset: {len(dataset)} samples")
print(f"  - Rollouts per prompt: {PROF_CONFIG['n_rollouts']}")
print(f"  - PROF filter removes: {PROF_CONFIG['prof_filter']}")
print(f"  - Max steps: {TRAIN_CONFIG['max_steps']}")
print("="*60 + "\n")

start_time = time.time()

try:
    trainer.train()
    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print(f"Total time: {(time.time() - start_time)/60:.1f} minutes")
    print("="*60)
except Exception as e:
    print(f"\nTraining error: {e}")
    raise

In [ ]:
# ============================================================
# CELL 13: Save Model
# ============================================================

output_path = "./prof-grpo-final"

print(f"Saving model to {output_path}")
trainer.save_model(output_path)
tokenizer.save_pretrained(output_path)

print(f"\nModel saved!")
print(f"Path: {output_path}")
print("\nTo use this model later:")
print(f"  model = AutoModelForCausalLM.from_pretrained('{output_path}')")

---

## Notes for P3 (Evaluation)

The saved model checkpoint is in `./prof-grpo-final/`. To evaluate:

1. **Pass@k evaluation**: Generate k samples per problem and check if any are correct
2. **RAC (Reward-Aware Coverage)**: Measure the coverage of correct reasoning steps
3. **Compare against baseline**: Run training again with `baseline_reward_fn` instead of `prof_reward_fn`

### Quick Evaluation Code:
```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("./prof-grpo-final")
tokenizer = AutoTokenizer.from_pretrained("./prof-grpo-final")

# Generate and evaluate on test set
# ...
```